# SGP Kit Balance Report

This report examines kit balance in this edition of regular **Soirée du Grand Poulet** play. Major events are excluded.

The figures compare adoption, exposure, combat output, ability performance, player concentration, matchups, and player-skill context. They are balance signals from observed FFA play, not controlled win-rate estimates.

In [ ]:
from pathlib import Path
from plotly.offline import init_notebook_mode

# Initialize Plotly offline mode for Jupyter Notebook
init_notebook_mode(connected=True)

from sgp_data import load_report_data
from sgp_report import (
    ability_effectiveness_figure,
    ability_uses_figure,
    damage_causes_figure,
    damage_figure,
    elo_adjusted_kill_results_figure,
    kill_causes_figure,
    kill_concentration_figure,
    kill_concentration_scatter_figure,
    kills_vs_ability_uses_figure,
    matchup_figure,
    player_reach_figure,
    popularity_efficiency_figure,
    show_cause_profile_figure,
    show_player_contribution_figure,
    top_killer_exposure_figure,
    total_kills_figure,
)


report = load_report_data(Path("data"))


# Kit adoption

## Player reach and exposure

The opening view compares the share of observed players who tried each kit, used its ability, and made at least one attributed player kill. The denominator is every player present in the extracted SGP data, so the gaps between markers show where broad adoption did—or did not—carry through to ability engagement and kill participation.

**Exposure shares** compares how the edition's playtime and completed lives were distributed across kits. **Life duration** shows average active time per completed life; unfinished current lives are not part of that denominator.

In [ ]:
fig = player_reach_figure(report)
fig.show()


# Kills

## Kill output and exchange

Use total kills to judge evidence volume, **By player** to see who produced it, and the hourly and per-life modes to compare output after exposure. A kit's aggregate rate is its total kills divided by its total active time or completed lives. The hover's median individual rate is instead the median of each eligible player's own rate; `n` gives the number of players included.

**Kills / PvP death** compares attributed kills with player-caused deaths while using the kit; the black line marks parity at 1. Non-player deaths are shown separately in hover. In the stacked view, click a player to focus that player across all kits, then click again to reset.

In [ ]:
fig = total_kills_figure(report)
show_player_contribution_figure(fig)


## Kill causes as offensive identity and defensive vulnerability

**Outgoing share** shows how each kit secures attributed player kills: a dominant cause suggests a narrow offensive identity, while a mixed profile points to several finishing mechanisms. It includes only kills with a known attacking and victim kit.

The incoming modes show defensive vulnerability. The hourly view preserves death-rate magnitude; the share view isolates the cause mix. They include non-player deaths when the victim kit is known. Click a cause to focus it across every kit and mode, then click again to reset.

In [ ]:
fig = kill_causes_figure(report)
show_cause_profile_figure(fig)


## Playtime share vs. kill efficiency

**Popularity** separates playtime share from kills per active hour. Its guides mark an equal share of kit playtime and the edition-wide kill rate. **Player Elo** replaces popularity with each kit's playtime-weighted player rating; its vertical guide is the overall playtime-weighted rating. This helps distinguish high output accompanied by a stronger player population from high output produced by a more typical field.

Both modes remain descriptive: FFA play style, map movement, and fight selection can affect kills per hour.

In [ ]:
fig = popularity_efficiency_figure(report)
fig.show()


## Player concentration of output and exposure

Each bar partitions a selected kit total into the top player, players 2–3, and everyone else. Switch among kills, attributed damage dealt, all damage received, playtime, and completed lives. The contributor count above each bar gives the context needed to interpret a concentrated result; the received-damage mode groups target players and includes non-player sources.

In [ ]:
fig = kill_concentration_figure(report)
fig.show()


## Total kills vs. player concentration

Each point is a kit. The median guides separate evidence volume from dependence on the top killer. High concentration deserves more caution when judging a kit from aggregate output, especially when few players contributed kills.

In [ ]:
fig = kill_concentration_scatter_figure(report)
fig.show()


## Leading-player output relative to exposure

Select either each kit's top killer or its most-played player. The point compares that player's share of kit playtime with the same player's share of kit kills. Above the diagonal, kill contribution exceeds exposure; below it, exposure exceeds kill contribution.

Use this alongside the concentration view to separate genuine single-player dependence from output that is broadly proportional to who spent the most time on the kit. Kits without valid shares are omitted.

In [ ]:
fig = top_killer_exposure_figure(report)
fig.show()


# Damage

Damage measures cumulative pressure in hearts, including damage that is later healed, so it complements rather than replaces the kill results.

## Damage output, intake, and player contribution

Total and hourly dealt modes measure attributed player pressure from each kit; **By player** shows who supplied it. Received damage includes every source when the target kit is known and separates player from non-player pressure. **PvP exchange** compares player-attributed damage dealt with player-attributed damage received; the black line marks parity at 1. Click a player segment to focus that player across all kits.

In [ ]:
fig = damage_figure(report)
show_player_contribution_figure(fig)


## Damage causes as offensive identity and defensive pressure

The dealt modes show which causes produce each kit's attributed player damage; the received modes show what pressures each target kit. Hourly modes retain pressure magnitude, while share modes isolate the mechanism mix. Click a cause to focus it across every kit and mode, then click again to reset.

In [ ]:
fig = damage_causes_figure(report)
show_cause_profile_figure(fig)


# Matchups

These views compare observed kills and attributed damage between kit pairs. They are not encounter win rates: fights without a logged kill and time spent in each matchup are not observed, and damage may later be healed.

## Observed head-to-head exchanges

In the share modes, a value above 50% means the row kit led the exchange against the column kit. **Kills vs Elo** shows observed kill share minus the share implied by the two players' current ratings; positive values favor the row kit after that player-skill context. The Elo expectation uses the extraction-time snapshot, not each player's historical pre-kill rating. Raw modes show the underlying directional kill and damage volume, which is essential when a striking share rests on little evidence.

In [ ]:
fig = matchup_figure(
    report.matchup_matrix,
    report.directional_share,
    report.pair_totals,
    report.matchup_kills_by_cause,
    elo_matchup_expected_share=report.elo_matchup_expected_share,
    elo_matchup_score_difference=report.elo_matchup_score_difference,
    elo_matchup_pair_totals=report.elo_matchup_pair_totals,
    elo_name=(
        str(report.elo_metadata["elo_name"].iloc[0])
        if not report.elo_metadata.empty
        else "Kill Elo"
    ),
    damage_matchup_matrix=report.damage_matchup_matrix,
    damage_directional_share=report.damage_directional_share,
    damage_pair_totals=report.damage_pair_totals,
    matchup_damage_by_cause=report.matchup_damage_by_cause,
)
fig.show()


# Abilities

Each kit has one logged ability. Activations measure engagement; successful uses and the kit-specific effect metric describe whether those activations achieved their logged purpose.

## Ability uses by kit

**Cooldown-normalized use** divides activations per active hour by the maximum rate implied by the configured cooldown. A value of 25% means roughly one activation per four cooldown lengths of active time. It is not a combat-opportunity rate: travel and downtime remain in the denominator, and cooldown resets can produce values above 100%.

A successful use follows the kit-specific condition shown in hover; it does not mean the ability caused a kill. Poseidon's Splash has no success or effect metric and is omitted only from unsupported modes. Rate hovers distinguish the kit aggregate from the median eligible player's own rate. Click a player segment to focus that player across all kits.

In [ ]:
fig = ability_uses_figure(report)
show_player_contribution_figure(fig)


## Ability engagement and effectiveness

The default mode separates how often an ability is activated from how often it meets its own success condition. The other modes compare effect magnitude only where the metadata provides a genuinely shared unit: affected players per successful cast, or health impact in hearts per successful use. Median guides and quadrants are recalculated for each eligible subset.

The hearts mode combines offensive damage for Tank and Cancer with resisted damage for Enderman. It is a common magnitude scale, not a claim that dealing and resisting damage have identical balance value. Pigeon's lock time, Archer's displacement, and Alchemist's destroyed decoys use unique units and should be interpreted within their own kit rather than ranked on the common axes.

In [ ]:
fig = ability_effectiveness_figure(report)
fig.show()


# Combined analysis

## Kills vs. ability uses

Each point is a kit. Switch among aggregate totals, cooldown-normalized ability use against kills per hour, successful-use rate against kills per hour, and per-completed-life rates; median lines and quadrant labels are recalculated for every mode. The usage modes answer a different question from the effectiveness plot above: whether observed ability engagement is associated with kill output. None of these comparisons implies that ability use or success causes kills.

In [ ]:
fig = kills_vs_ability_uses_figure(report.combined_totals)
fig.show()


## Kill results relative to current player Elo

Each credited cross-kit PvP kill is treated as a binary result: one score for the killer's kit and zero for the victim's kit. The plot compares each kit's observed score share with the score implied by the two players' current Kill Elo ratings, then sorts kits by the difference. Same-kit kills are excluded because they add one win and one loss to the same kit and carry no kit-balance signal.

Treat this as player-skill context, not a reconstructed historical expectation: the extraction contains current ratings rather than each player's rating immediately before every kill. Large differences based on few cross-kit kills or deaths should be interpreted cautiously.

In [ ]:
fig = elo_adjusted_kill_results_figure(report)
fig.show()
